In [2]:
import csv
import json
import requests
from pathlib import Path
from tqdm import tqdm
import re

In [9]:
# -------------------------------
# USER CONFIGURATION
# -------------------------------
LOW_QUALITY_TAGS = ["course", "teaching", "education", "homework"]
API_URL = "http://10.52.88.30:11434/api/generate"
MODEL_NAME = "gpt-oss:20b"
INPUT_CSV = "./hq/biostars/full/biostars_db.csv"
GOOD_JSON = "./db/metageneassistant_clean.jsonl"
REVIEW_JSON = "./db/metageneassistant_review.jsonl"

MIN_SCORE = 1
ONLY_ACCEPTED = False
MAX_TOKENS = 2048
TIMEOUT = 60

In [10]:
def extract_json_block(text):
    if not text:
        return None
        
    # Remove markdown ```json fences
    text = text.replace("```json", "").replace("```", "")

    # Regex to capture the FIRST {...} block
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        return None

    json_str = match.group(0)

    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        return None

In [11]:
def call_llm(prompt):
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "max_tokens": MAX_TOKENS,
        "temperature": 0.0
    }

    try:
        response = requests.post(API_URL, json=payload, stream=True, timeout=TIMEOUT)
        # print("STATUS:", response.status_code)

        if response.status_code != 200:
            print("Bad HTTP response:", response.text)
            return ""

        final_text = ""

        # Ollama returns NDJSON (JSON objects per line)
        for line in response.iter_lines():
            if not line:
                continue

            try:
                obj = json.loads(line.decode("utf-8"))
            except:
                print("Skipping unparsable chunk:", line)
                continue

            # Skip thinking / token streaming
            if "response" in obj and obj.get("response"):
                final_text += obj["response"]

            # Stop when done
            if obj.get("done", False):
                break

        return final_text.strip()

    except Exception as e:
        print("LLM error:", e)
        return ""

In [16]:
def build_prompt(question, answer, tags):
   return f"""
You are a STRICT data-cleaning and data-structuring model.
You must ONLY use information explicitly present in the QUESTION and ANSWER below.
You MUST NOT add, infer, explain, guess, or hallucinate any extra information.

Your tasks:

1. DOMAIN CHECK (STRICT)
   - If the Q/A is NOT clearly about computational biology, bioinformatics, sequencing, genomics, metagenomics, pipelines, tools, or data analysis → output: {{ "valid": false, "reason": "out_of_domain" }}.

2. QUALITY CHECK (STRICT)
   - If the ANSWER is missing, incomplete, off-topic, or does not address the question → output: {{ "valid": false, "reason": "low_quality" }}.

3. CLASSIFICATION
   Only classify based on what is EXPLICITLY in the Q/A:
   - "technical_troubleshooting" → problems, errors, bugs, tool failures, contamination, mapping issues, adapter issues, etc.
   - "factual_conceptual" → definitions, explanations, conceptual clarifications.
   - "workflow_design" → pipeline steps, protocols, recommended procedures.

4. TRANSFORMATION RULES (NO INVENTION)
   Using ONLY the text from QUESTION and ANSWER:
   - instruction = restate the QUESTION as a clear instruction using ONLY its content.
   - input = summarize any technical logs, parameters, error messages, tool names FOUND in the QUESTION (do not add new ones).
   - output = rewrite the ANSWER cleanly, using EXACTLY and ONLY the ideas present in the original ANSWER.
   NO NEW DETAILS. NO external knowledge. NO assumptions.

5. OUTPUT RULES (VERY IMPORTANT)
   - Your response MUST be exactly ONE JSON object.
   - NO explanation before or after.
   - NO markdown.
   - NO code fences.
   - NO extra commentary.

JSON format:

{{
  "valid": true/false,
  "category": "...",
  "instruction": "...",
  "input": "...",
  "output": "...",
  "reason": "..."
}}

QUESTION:
{question}

ANSWER:
{answer}

TAGS:
{tags}

Return ONLY the JSON.
"""

In [ ]:
LINES_TO_SKIP = 5000

with open(GOOD_JSON, "a", encoding="utf-8") as good_out, \
     open(REVIEW_JSON, "a", encoding="utf-8") as bad_out, \
     open(INPUT_CSV, newline='', encoding='utf-8') as f:

    # for _ in range(LINES_TO_SKIP):
    #     try:
    #         next(f)
    #     except StopIteration:
    #         print(f"File ended unexpectedly after skipping {_+1} lines.")
        
    reader = csv.DictReader(f)

    for index, row in tqdm(enumerate(reader), desc="Filtering Data: "): 
        if index < LINES_TO_SKIP:
            continue
        
        # ---- extract fields ----
        question = row.get("question", "").strip()
        answer = row.get("answer", "").strip()
        score = int(row.get("score", "0"))
        accepted = row.get("accepted", "").lower() == "true"
        tags = row.get("tags", "")
        url = row.get("url", "")

        # ---- simple filtering ----
        if score < MIN_SCORE:
            row["reason"] = "low_score"
            bad_out.write(json.dumps(row) + "\n")
            continue

        if ONLY_ACCEPTED and not accepted:
            row["reason"] = "not_accepted"
            bad_out.write(json.dumps(row) + "\n")
            continue

        tag_list = [t.strip().lower() for t in tags.split(",")]
        if any(bad_tag in tag_list for bad_tag in LOW_QUALITY_TAGS):
            row["reason"] = "irrelevant_tags"
            bad_out.write(json.dumps(row) + "\n")
            continue

        # ---- call LLM ----
        prompt = build_prompt(question, answer, tags)
        llm_response = call_llm(prompt)

        parsed = extract_json_block(llm_response)
        if not parsed:
            row["reason"] = "llm_parse_error"
            bad_out.write(json.dumps(row) + "\n")
            continue

        # ---- categorize ----
        if not parsed.get("valid", False):
            row["reason"] = parsed.get("reason", "invalid")
            bad_out.write(json.dumps(row) + "\n")
            continue

        # ---- save cleaned ----
        cleaned = {
            "instruction": parsed.get("instruction", "").strip(),
            "input": parsed.get("input", "").strip(),
            "output": parsed.get("output", "").strip(),
            "metadata": {
                "tags": tags,
                "category": parsed.get("category", "").strip(),
                "model": MODEL_NAME,
                "source": {
                    "url": url,
                    "question": question,
                    "answer": answer,
                }
            },
        }

        good_out.write(json.dumps(cleaned) + "\n")
        # break


In [13]:
#### SECOND PASS CHECK

In [ ]:
# --------- FILES ---------
REVIEW_ROUND1 = "./db/metageneassistant_review.jsonl"
REVIEW_ROUND2 = "./db/metageneassistant_review_R2.jsonl"
GOOD_JSON = "./db/metageneassistant_clean_R2.jsonl"
MODEL_NAME = "gpt-oss:20b"

kept = 0
flagged = 0
total = 0

with open(REVIEW_ROUND1, "r", encoding="utf-8") as f_in, \
        open(GOOD_JSON, "a", encoding="utf-8") as good_out, \
        open(REVIEW_ROUND2, "w", encoding="utf-8") as bad_out:

    for line in tqdm(f_in, desc="Second Pass Review"):
        total += 1
        row = json.loads(line)

        question = row.get("question", "").strip()
        answer   = row.get("answer", "").strip()
        tags     = row.get("tags", "")
        url      = row.get("url", "")
        score = int(row.get("score", "0"))
        accepted = row.get("accepted", "").lower() == "true"
        
        # ---- simple filtering ----
        if score < MIN_SCORE:
            row["reason"] = "low_score"
            bad_out.write(json.dumps(row) + "\n")
            continue

        if ONLY_ACCEPTED and not accepted:
            row["reason"] = "not_accepted"
            bad_out.write(json.dumps(row) + "\n")
            continue

        tag_list = [t.strip().lower() for t in tags.split(",")]
        if any(bad_tag in tag_list for bad_tag in LOW_QUALITY_TAGS):
            row["reason"] = "irrelevant_tags"
            bad_out.write(json.dumps(row) + "\n")
            continue

        # ---------------- LLM CHECK AGAIN ----------------
        prompt = build_prompt(question, answer, tags)
        llm_response = call_llm(prompt)

        parsed = extract_json_block(llm_response)
        if not parsed:
            row["reason"] = "llm_parse_error_round2"
            bad_out.write(json.dumps(row) + "\n")
            flagged += 1
            continue

        if not parsed.get("valid", False):
            row["reason"] = parsed.get("reason", "invalid_round2")
            bad_out.write(json.dumps(row) + "\n")
            flagged += 1
            continue

        # ---------------- ACCEPT & CLEAN ----------------
        cleaned = {
            "instruction": parsed.get("instruction", "").strip(),
            "input": parsed.get("input", "").strip(),
            "output": parsed.get("output", "").strip(),
            "metadata": {
                "tags": tags,
                "category": parsed.get("category", "").strip(),
                "model": MODEL_NAME,
                "source": {
                    "url": url,
                    "question": question,
                    "answer": answer,
                }
            }
        }

        good_out.write(json.dumps(cleaned) + "\n")
        kept += 1

print("\n========= Second-Pass Summary ============")
print(f"Total rows reviewed: {total}")
print(f"Accepted and moved to clean: {kept}")
print(f"Still flagged (round 2): {flagged}")
print("==========================================\n")


Second Pass Review: 102it [15:57,  8.01s/it]

LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Read timed out.
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc26251b310>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc261f36d50>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc26202bad0>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/gen

Second Pass Review: 117it [15:57,  1.79s/it]

LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc26347bfd0>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc26251ab10>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc261f37a50>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object 

Second Pass Review: 132it [15:58,  1.43it/s]

...

Second Pass Review: 1361it [16:15, 70.82it/s]

LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc261fcd650>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc2801c4b90>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc261fcf010>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object 

Second Pass Review: 1369it [16:15, 70.34it/s]

LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc2624fb010>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc261dadf90>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fc261f53650>: Failed to establish a new connection: [Errno 111] Connection refused'))
LLM error: HTTPConnectionPool(host='10.52.88.30', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object 

Second Pass Review: 7785it [2:10:32, 10.49s/it]